In [ ]:
# Import libraries
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import math
import glob

# ----------------------------------------------------
# Global paths and file references
# ----------------------------------------------------
# For non-O3 variables, still use the old merged file
MERGED_FILE = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc'

# New directory for O3 data (200 years, each file contains one year's worth of data)
CLIM_DIR = '/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/atm/hist/h1/O3'

# ----------------------------------------------------
# Data Processing Functions
# ----------------------------------------------------
def process_variable(ds, var):
    """
    Apply the specific transformation for O3, T, U:
      - O3: first take the longitude mean, then select latitudes 60-90°N, apply a cosine latitude weighting,
            and average over latitude. Also, if the vertical dimension is named 'lev', rename it to 'plev'.
      - T: take lon-mean, then select latitudes 60-90°N and take the minimum along latitude.
      - U: take lon-mean, then select nearest latitude at 60°N.
    """
    if var == 'O3':
        da = ds['O3'].mean(dim='lon')
        if 'member' in da.dims:
            da = da.mean(dim='member')
        da = da.sel(lat=slice(60, 90))
        weights = np.cos(np.deg2rad(da.lat))
        da = da.weighted(weights).mean(dim='lat')
        # For new O3 files, the vertical dimension is 'lev'; rename it to 'plev' for consistency
        if 'lev' in da.dims:
            da = da.rename({'lev': 'plev'})
        return da
    elif var == 'T':
        da = ds['T'].mean(dim='lon')
        if 'member' in da.dims:
            da = da.mean(dim='member')
        da = da.sel(lat=slice(60, 90)).min(dim='lat')
        return da
    elif var == 'U':
        da = ds['U'].mean(dim='lon')
        if 'member' in da.dims:
            da = da.mean(dim='member')
        da = da.sel(lat=60, method='nearest')
        return da
    else:
        raise ValueError("Unsupported variable: " + var)

def create_monthly_means(da, start_month=1, end_month=12):
    """
    Select time within the given month range, group by month, compute the mean over time,
    and assign a dummy time coordinate (2000-MM-15) for plotting purposes.
    Returns (time_coord, plev_vals, data_array).
    """
    da = da.sel(time=da.time.dt.month.isin(range(start_month, end_month+1)))
    da_month = da.groupby('time.month').mean(dim='time')
    da_month = da_month.sortby('month')
    months = da_month['month'].values
    dummy_time = np.array([np.datetime64(f'2000-{int(m):02d}-15') for m in months])
    da_month = da_month.rename({'month': 'time'})
    da_month = da_month.assign_coords(time=dummy_time)
    da_month = da_month.transpose('plev', 'time')
    plev_vals = da_month['plev'].values
    time_vals = da_month['time'].values
    data_array = da_month.values
    return time_vals, plev_vals, data_array

def read_climatology(var):
    """
    For O3: Read all new O3 nc files (covering 200 years), merge them by time, 
    then process the variable and compute the multi-year monthly mean (i.e. the climatology).
    For T and U: use the original merged file.
    Returns an xarray DataArray with dims (plev, time) where 'time' is a dummy coordinate for months.
    """
    if var == 'O3':
        file_pattern = os.path.join(CLIM_DIR, "CO2x1SmidEmin_yBWCN.cam.h1.*.O3.isobar.nc")
        # Merge all files along the time dimension
        ds = xr.open_mfdataset(file_pattern, combine='by_coords')
        # (xarray will decode time using the provided units and calendar, e.g., "days since 0001-01-01")
        da = process_variable(ds, var)
        ds.close()
        # Compute the climatology (monthly mean over all available years)
        time_vals, plev_vals, data_arr = create_monthly_means(da, 1, 12)
        clim_da = xr.DataArray(data_arr,
                               coords={'plev': plev_vals, 'time': time_vals},
                               dims=['plev', 'time'])
        return clim_da
    else:
        ds = xr.open_dataset(MERGED_FILE)
        da = process_variable(ds, var)
        ds.close()
        time_vals, plev_vals, data_arr = create_monthly_means(da, 1, 12)
        clim_da = xr.DataArray(data_arr,
                               coords={'plev': plev_vals, 'time': time_vals},
                               dims=['plev', 'time'])
        return clim_da

# ----------------------------------------------------
# Plotting Functions
# ----------------------------------------------------
def plot_climatology(clim_da, var, save_path):
    """
    Plot the climatology (monthly mean vertical profile) for a given variable.
    """
    time_mpl = mdates.date2num(clim_da.time.values)
    fig, ax = plt.subplots(figsize=(8,6))
    cf = ax.contourf(time_mpl, clim_da.plev.values, clim_da.values, levels=20,
                     cmap='RdBu_r')
    ax.set_yscale('log')
    ax.invert_yaxis()
    ax.set_xlim(time_mpl[0], time_mpl[-1])
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.set_ylabel('Pressure (Pa)', fontsize=10)
    ax.set_title(f"{var} Climatology (200-year Mean)", fontsize=12)
    cbar = fig.colorbar(cf, ax=ax, label=f'{var} Climatology')
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"[INFO] Saved {var} climatology figure: {save_path}")

# ----------------------------------------------------
# For T and U, original functions for reading year-specific data and experiments remain
# ----------------------------------------------------
def read_year_data(var, year, start_month=1, end_month=12):
    """
    Read the data for a specific year and process the variable from the merged file (for T and U).
    Returns an xarray DataArray with dims (plev, time).
    """
    ds = xr.open_dataset(MERGED_FILE)
    ds_year = ds.sel(time=ds.time.dt.year == int(year))
    da = process_variable(ds_year, var)
    ds.close()
    time_vals, plev_vals, data_arr = create_monthly_means(da, start_month, end_month)
    year_da = xr.DataArray(data_arr,
                           coords={'plev': plev_vals, 'time': time_vals},
                           dims=['plev', 'time'])
    return year_da

def read_experiment_data(var, file_pattern, start_month=1, end_month=5):
    """
    Open the experiment file(s) from the old source, process the variable,
    then create monthly means for the specified range.
    Returns an xarray DataArray with dims (plev, time).
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    da = process_variable(ds, var)
    ds.close()
    time_vals, plev_vals, data_arr = create_monthly_means(da, start_month, end_month)
    exp_da = xr.DataArray(data_arr,
                          coords={'plev': plev_vals, 'time': time_vals},
                          dims=['plev', 'time'])
    return exp_da

def get_anomaly(data_da, clim_da):
    """
    Compute anomaly = data_da - clim_da by aligning the dummy time coordinate (month).
    """
    data_months = [int(str(t)[5:7]) for t in data_da.time.values]
    clim_months = [int(str(t)[5:7]) for t in clim_da.time.values]
    aligned_vals = np.zeros_like(data_da.values)
    for i, m in enumerate(data_months):
        j = clim_months.index(m)
        aligned_vals[:, i] = data_da.values[:, i] - clim_da.values[:, j]
    anom_da = xr.DataArray(aligned_vals,
                           coords={'plev': data_da.plev, 'time': data_da.time},
                           dims=['plev', 'time'])
    return anom_da

# ----------------------------------------------------
# Plotting function for composite experiments (for T and U only)
# ----------------------------------------------------
def auto_subplot_layout(n):
    cols = int(math.ceil(math.sqrt(n)))
    rows = int(math.ceil(n / cols))
    return rows, cols

def determine_common_months(runs):
    if not runs:
        return [3, 4, 5]
    start_months = []
    for run in runs:
        if len(run['time']) > 0:
            m = int(str(run['time'][0])[5:7])
            start_months.append(m)
    if not start_months:
        return [3, 4, 5]
    latest_start = max(start_months)
    return list(range(latest_start, 6))

def trim_to_months(run, months):
    if not months:
        return run
    keep_indices = []
    for i, t in enumerate(run['time']):
        m = int(str(t)[5:7])
        if m in months:
            keep_indices.append(i)
    run['time'] = run['time'][keep_indices]
    run['data'] = run['data'][:, keep_indices]
    return run

def plot_composite(runs, ref_run, var, composite_name, year, save_path):
    all_runs = runs + [ref_run]
    common_months = determine_common_months(all_runs)
    for r in all_runs:
        trim_to_months(r, common_months)
    all_anom = []
    for r in all_runs:
        all_anom.append(r['data'].ravel())
    all_anom = np.concatenate(all_anom)
    vmax = np.nanmax(np.abs(all_anom))
    vmin = -vmax
    total_subplots = len(all_runs)
    rows, cols = auto_subplot_layout(total_subplots)
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows), squeeze=False)
    subplot_idx = 0
    cf = None
    for run in all_runs:
        r_idx = subplot_idx // cols
        c_idx = subplot_idx % cols
        ax = axes[r_idx][c_idx]
        if len(run['time']) > 0:
            time_mpl = mdates.date2num(run['time'])
            cf = ax.contourf(time_mpl, run['plev'], run['data'], levels=20,
                             cmap='RdBu_r', vmin=vmin, vmax=vmax)
            ax.set_yscale('log')
            ax.invert_yaxis()
            ax.set_xlim(time_mpl[0], time_mpl[-1])
            ax.xaxis.set_major_locator(mdates.MonthLocator())
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
        ax.set_ylabel('Pressure (Pa)', fontsize=8)
        ax.set_title(f"{run['label']} (Year {run['year']})", fontsize=10)
        subplot_idx += 1
    for i in range(subplot_idx, rows*cols):
        r_idx = i // cols
        c_idx = i % cols
        axes[r_idx][c_idx].axis('off')
    if cf is not None:
        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        fig.colorbar(cf, cax=cax, label=f'{var} Anomaly')
    fig.suptitle(f"{composite_name} - {var} Anomaly\nYear {year} (Latest start month to May)", fontsize=14)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"[INFO] Saved composite figure: {save_path}")

# ----------------------------------------------------
# Define composite groups for T and U experiments (O3 is handled separately as climatology)
# ----------------------------------------------------
def get_year_small_pert(year):
    base = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}'
    file_Feb = os.path.join(base, 'Feb', f'BWCN.e122.f19_g16.002.{year}-02.*.nc')
    file_Mar = os.path.join(base, 'Mar', f'BWCN.e122.f19_g16.002.{year}-03.*.nc')
    return file_Feb, file_Mar

def get_year_nocouple(year):
    file_Feb = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    file_Mar = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    return file_Feb, file_Mar

file_0008_Jan_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Jan/BWCN.e122.f19_g16.002.0008-01.*.nc'
file_0008_Feb_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc'
file_0008_Feb_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_2/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_2/BWCN.e122.f19_g16.002.0008-03.*.nc'
file_0008_Apr_large = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Apr/BWCN.e122.f19_g16.002.0008-04.*.nc'
file_0008_Feb_moist = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb_3/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_moist = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar_3/BWCN.e122.f19_g16.002.0008-03.*.nc'

composite_groups = [
    {'name': 'Composite 1', 'year': '0008', 'runs': [
        {'label': 'Jan small pertlim', 'file': file_0008_Jan_small},
        {'label': 'Feb small pertlim', 'file': file_0008_Feb_small},
        {'label': 'Mar small pertlim', 'file': file_0008_Mar_small}
    ]},
    {'name': 'Composite 2', 'year': '0008', 'runs': [
        {'label': 'Feb large pertlim', 'file': file_0008_Feb_large},
        {'label': 'Mar large pertlim', 'file': file_0008_Mar_large},
        {'label': 'Apr large pertlim', 'file': file_0008_Apr_large}
    ]},
    {'name': 'Composite 3', 'year': '0008', 'runs': [
        {'label': 'Feb small pertlim', 'file': file_0008_Feb_small},
        {'label': 'Feb large pertlim', 'file': file_0008_Feb_large},
        {'label': 'Feb moist pertlim', 'file': file_0008_Feb_moist}
    ]},
    {'name': 'Composite 4', 'year': '0008', 'runs': [
        {'label': 'Mar small pertlim', 'file': file_0008_Mar_small},
        {'label': 'Mar large pertlim', 'file': file_0008_Mar_large},
        {'label': 'Mar moist pertlim', 'file': file_0008_Mar_moist}
    ]},
    {'name': 'Composite 5', 'year': '0003', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0003')[0]},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0003')[1]}
    ]},
    {'name': 'Composite 6', 'year': '0013', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0013')[0]},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0013')[1]}
    ]},
    {'name': 'Composite 7', 'year': '0014', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0014')[0]},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0014')[1]}
    ]},
    {'name': 'Composite 8', 'year': '0019', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0019')[0]},
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0019')[1]}
    ]},
    {'name': 'Composite 9', 'year': '0008', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc'},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc'}
    ]},
    {'name': 'Composite 10', 'year': '0013', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Feb_NOCOUPL/0013-02/*.nc'},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Mar_NOCOUPL/0013-03/*.nc'}
    ]},
    {'name': 'Composite 11', 'year': '0014', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Feb_NOCOUPL/0014-02/*.nc'},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Mar_NOCOUPL/0014-03/*.nc'}
    ]},
    {'name': 'Composite 12', 'year': '0019', 'runs': [
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Feb_NOCOUPL/0019-02/*.nc'},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Mar_NOCOUPL/0019-03/*.nc'}
    ]},
    {'name': 'Composite 13', 'year': '0008', 'runs': [
        {'label': 'Feb small pertlim', 'file': file_0008_Feb_small},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc'}
    ]},
    {'name': 'Composite 14', 'year': '0008', 'runs': [
        {'label': 'Mar small pertlim', 'file': file_0008_Mar_small},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc'}
    ]},
    {'name': 'Composite 15', 'year': '0013', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0013')[0]},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Feb_NOCOUPL/0013-02/*.nc'}
    ]},
    {'name': 'Composite 16', 'year': '0013', 'runs': [
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0013')[1]},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0013/Mar_NOCOUPL/0013-03/*.nc'}
    ]},
    {'name': 'Composite 17', 'year': '0014', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0014')[0]},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Feb_NOCOUPL/0014-02/*.nc'}
    ]},
    {'name': 'Composite 18', 'year': '0014', 'runs': [
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0014')[1]},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0014/Mar_NOCOUPL/0014-03/*.nc'}
    ]},
    {'name': 'Composite 19', 'year': '0019', 'runs': [
        {'label': 'Feb small pertlim', 'file': get_year_small_pert('0019')[0]},
        {'label': 'Feb nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Feb_NOCOUPL/0019-02/*.nc'}
    ]},
    {'name': 'Composite 20', 'year': '0019', 'runs': [
        {'label': 'Mar small pertlim', 'file': get_year_small_pert('0019')[1]},
        {'label': 'Mar nocouple', 'file': '/home/weiji/restart_exam/nocouple_data/0019/Mar_NOCOUPL/0019-03/*.nc'}
    ]}
]

def generate_all_composites():
    output_dir = '/home/weiji/restart_exam/20250221/NEWO3CLIMVERTICAL/'
    variables = ['O3', 'T', 'U']

    for var in variables:
        print(f"\n[INFO] Generating composite plots for {var}...")
        if var == 'O3':
            # For O3, only compute the climatology from the new merged 200-year data
            clim_da = read_climatology('O3')
            save_path = os.path.join(output_dir, "O3_Climatology.png")
            plot_climatology(clim_da, var, save_path)
        else:
            # For T and U, use original procedure: compute ref anomaly and composite experiments
            clim_da = read_climatology(var)
            for group in composite_groups:
                year = group['year']
                composite_name = group['name']
                runs_info = group['runs']
                year_da = read_year_data(var, year, 1, 12)
                ref_anom = get_anomaly(year_da, clim_da)
                ref_run = {
                    'label': 'REF',
                    'year': year,
                    'time': ref_anom.time.values,
                    'plev': ref_anom.plev.values,
                    'data': ref_anom.values
                }
                exp_runs = []
                for r in runs_info:
                    exp_da = read_experiment_data(var, r['file'], 1, 5)
                    exp_anom = get_anomaly(exp_da, clim_da)
                    exp_runs.append({
                        'label': r['label'],
                        'year': year,
                        'time': exp_anom.time.values,
                        'plev': exp_anom.plev.values,
                        'data': exp_anom.values
                    })
                save_path = os.path.join(output_dir, f"{composite_name.replace(' ', '_')}_{var}.png")
                plot_composite(exp_runs, ref_run, var, composite_name, year, save_path)

    print("\n[INFO] All composite plots have been generated.")

if __name__ == '__main__':
    generate_all_composites()
